# W&B Rerun — Runner 7 of 8

**Assignment:** 3 Heavy (P.30.x combination experiments)  
**Experiments:** 3  

1. **vR.P.30** -- Multi-Q ELA + CBAM (25ep) (run01)  
2. **vR.P.30.1** -- Multi-Q ELA + CBAM (50ep) (run01)  
3. **vR.P.30.2** -- Multi-Q ELA + CBAM + Unfreeze (40ep) (run01)

Each notebook runs sequentially via **papermill**, logging metrics
to the `tamper-detection-ablation` W&B project in real-time.


In [ ]:
# ============================================================
# Runner 7 of 8 -- Setup
# ============================================================
!pip install -q papermill wandb segmentation-models-pytorch albumentations

import os, gc, time, json
import torch
import papermill as pm

RUNNER_ID = 7
DATASET_PATH = "/kaggle/input/vrpx-source-notebooks"
OUTPUT_DIR = "/kaggle/working/executed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({mem:.1f} GB)')
else:
    print('WARNING: No GPU detected!')

# Verify source notebooks exist
if os.path.isdir(DATASET_PATH):
    files = [f for f in os.listdir(DATASET_PATH) if f.endswith('.ipynb')]
    print(f'Source dataset: {len(files)} notebooks found')
else:
    print(f'ERROR: Dataset not found at {DATASET_PATH}')
    print('Upload all vR.P.x notebooks as a Kaggle dataset named "vrpx-source-notebooks"')


In [ ]:
# ============================================================
# Runner 7 -- Experiment Queue (3 experiments)
# ============================================================
EXPERIMENTS = [
    {"file": "vR.P.30 Image Detection and Localisation.ipynb", "version": "vR.P.30", "desc": "Multi-Q ELA + CBAM (25ep)", "run_id": "run01"},
    {"file": "vR.P.30.1 Image Detection and Localisation.ipynb", "version": "vR.P.30.1", "desc": "Multi-Q ELA + CBAM (50ep)", "run_id": "run01"},
    {"file": "vR.P.30.2 Image Detection and Localisation.ipynb", "version": "vR.P.30.2", "desc": "Multi-Q ELA + CBAM + Unfreeze (40ep)", "run_id": "run01"},
]

print(f'\nRunner {RUNNER_ID}: {len(EXPERIMENTS)} experiments queued')
print('=' * 60)
for i, exp in enumerate(EXPERIMENTS, 1):
    src = os.path.join(DATASET_PATH, exp['file'])
    exists = 'OK' if os.path.exists(src) else 'MISSING'
    print(f'  {i}. [{exists:>7}] {exp["version"]:12} {exp["desc"]} ({exp["run_id"]})')


In [ ]:
# ============================================================
# Runner 7 -- Sequential Execution
# ============================================================
results = []
run_start = time.time()

for idx, exp in enumerate(EXPERIMENTS, 1):
    nb_path = os.path.join(DATASET_PATH, exp['file'])
    out_name = f"{exp['version'].replace('.', '')}_{exp['run_id']}_output.ipynb"
    out_path = os.path.join(OUTPUT_DIR, out_name)

    print(f'\n{"=" * 70}')
    print(f'  [{idx}/{len(EXPERIMENTS)}] {exp["version"]} -- {exp["desc"]} ({exp["run_id"]})')
    print(f'{"=" * 70}')

    if not os.path.exists(nb_path):
        print(f'  SKIP: {nb_path} not found')
        results.append({**exp, 'status': 'NOT_FOUND', 'time_min': 0})
        continue

    t0 = time.time()
    try:
        pm.execute_notebook(
            nb_path,
            out_path,
            kernel_name='python3',
            cwd='/kaggle/working',
        )
        elapsed = (time.time() - t0) / 60
        print(f'  SUCCESS in {elapsed:.1f} min')
        results.append({**exp, 'status': 'SUCCESS', 'time_min': round(elapsed, 1)})

    except pm.PapermillExecutionError as e:
        elapsed = (time.time() - t0) / 60
        err_msg = f'{e.ename}: {str(e.evalue)[:200]}'
        print(f'  FAILED after {elapsed:.1f} min')
        print(f'  {err_msg}')
        results.append({**exp, 'status': 'FAILED', 'time_min': round(elapsed, 1), 'error': err_msg})

    except Exception as e:
        elapsed = (time.time() - t0) / 60
        err_msg = str(e)[:200]
        print(f'  ERROR after {elapsed:.1f} min: {err_msg}')
        results.append({**exp, 'status': 'ERROR', 'time_min': round(elapsed, 1), 'error': err_msg})

    # -- GPU cleanup between experiments --
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print(f'  GPU cache cleared')

total_min = (time.time() - run_start) / 60
print(f'\nAll experiments processed. Total wall time: {total_min:.1f} min ({total_min/60:.1f} h)')


In [ ]:
# ============================================================
# Runner 7 -- Execution Summary
# ============================================================
print(f'\n{"=" * 70}')
print(f'  RUNNER 7 COMPLETE')
print(f'{"=" * 70}\n')

gpu_total = sum(r['time_min'] for r in results)
ok = [r for r in results if r['status'] == 'SUCCESS']
fail = [r for r in results if r['status'] != 'SUCCESS']

for r in results:
    icon = 'OK' if r['status'] == 'SUCCESS' else r['status']
    print(f"  [{icon:>9}] {r['version']:12} {r['desc']:40} {r['time_min']:6.1f} min")
    if 'error' in r:
        print(f"             {r['error'][:100]}")

print(f'\n  Succeeded: {len(ok)}/{len(results)}')
if fail:
    print(f'  Failed:    {len(fail)}/{len(results)}')
print(f'  GPU time:  {gpu_total:.1f} min ({gpu_total/60:.1f} h)')

# Save JSON summary
summary_path = os.path.join(OUTPUT_DIR, 'runner_7_summary.json')
with open(summary_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'\n  Summary saved: {summary_path}')

# List output notebooks
print(f'\n  Output notebooks:')
for f_name in sorted(os.listdir(OUTPUT_DIR)):
    if f_name.endswith('.ipynb'):
        size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f_name)) / 1e6
        print(f'    {f_name} ({size_mb:.1f} MB)')
